# Langchain Chatbot

### Install requirements

In [18]:
%pip install -U langchain langchain-community pypdf pytube langchain-openai openai python-dotenv youtube-transcript-api yt_dlp pydub

Note: you may need to restart the kernel to use updated packages.


### Imports and Envrionment

In [15]:
import os
import openai
import sys

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

api_key = os.environ.get("OPENAI_API_KEY")

## Document Loading

Document loaders access data from a variety of data sources and load them into a standard object. There are a lot of data loaders in Langchain (e.g., YouTube, PowerPoint, Figma, Notion, Excel, Pandas, Amazon AWS).

### Load PDF Documents

First let's load some scientific pdfs we downloaded from the web about marathon running. 

In [16]:
from langchain.document_loaders import PyPDFLoader

PDF_DIR = 'pdfs'

all_pages = []

print(os.getcwd())

for filename in os.listdir(PDF_DIR):
    filepath = os.path.join(PDF_DIR, filename)
    loader = PyPDFLoader(filepath)
    pages = loader.load()
    all_pages.extend(pages)

print(f"Loaded {len(all_pages)} pages.")

# Inspect first page
page = all_pages[0]
print(page.page_content[:500])
print(page.metadata)

/Users/jonas/Repositories/jonas-ml-lab/notebooks/projects/marathon coach
Loaded 250 pages.
Vol.:(0123456789)
Sports Medicine (2024) 54:1801–1833 
https://doi.org/10.1007/s40279-024-02018-z
SYSTEMATIC REVIEW
The Effect of Strength Training Methods on Middle‑Distance 
and Long‑Distance Runners’ Athletic Performance: A Systematic 
Review with Meta‑analysis
Cristian Llanos‑Lagos1  · Rodrigo Ramirez‑Campillo2 · Jason Moran3 · Eduardo Sáez de Villarreal1
Accepted: 10 March 2024 / Published online: 17 April 2024 
© The Author(s) 2024
Abstract
Background The running performance of middle-dist
{'producer': 'PyPDF', 'creator': 'Springer', 'creationdate': '2024-04-16T17:19:55+05:30', 'author': 'Cristian Llanos-Lagos', 'crossmarkdomains[1]': 'springer.com', 'crossmarkdomains[2]': 'springerlink.com', 'crossmarkdomainexclusive': 'true', 'crossmarkmajorversiondate': '2010-04-23', 'moddate': '2024-07-10T21:13:03+05:30', 'subject': 'Sports Medicine, https://doi.org/10.1007/s40279-024-02018-z', 'title':

In [19]:
from langchain_community.document_loaders import YoutubeLoader, YoutubeAudioLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import OpenAIWhisperParser
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

AUDIO_DIR = "audio"

URLS = [
    "https://www.youtube.com/watch?v=QEtIxBL497U",
]

# Pro: Fast and no API costs; Con: Only works if the video has transcripts
# def load_via_youtube_api(url: str):
#     loader = YoutubeLoader.from_youtube_url(
#         url, add_video_info=False, language=["en", "en-US"]
#     )
#     return loader.load()

# Uses OpenAI Whisper API (paid). Requires OPENAI_API_KEY and ffmpeg installed.
def load_via_OpenAI(url: str):
    audio_loader = YoutubeAudioLoader([url], AUDIO_DIR)
    parser = OpenAIWhisperParser()  # uses OPENAI_API_KEY from env
    loader = GenericLoader(audio_loader, parser)
    return loader.load()

yt_pages = []
for url in URLS:
    docs = load_via_OpenAI(url)
    yt_pages.extend(docs)

print(f"Loaded {len(yt_pages)} pages from YouTube.")

# Inspect first page (if any)
if yt_pages:
    page = yt_pages[0]
    print(page.page_content[:500])
    print(page.metadata)
else:
    print("No content loaded. Check URLs, transcripts availability, or Whisper setup.")

[youtube] Extracting URL: https://www.youtube.com/watch?v=QEtIxBL497U
[youtube] QEtIxBL497U: Downloading webpage
[youtube] QEtIxBL497U: Downloading tv simply player API JSON
[youtube] QEtIxBL497U: Downloading tv client config
[youtube] QEtIxBL497U: Downloading tv player API JSON
[info] QEtIxBL497U: Downloading 1 format(s): 140
[download] audio/The Smartest Way to Run a Faster Marathon (Science Explained).m4a has already been downloaded
[download] 100% of   17.53MiB
[ExtractAudio] Not converting audio audio/The Smartest Way to Run a Faster Marathon (Science Explained).m4a; file is already in target format m4a
Transcribing part 1!
Loaded 1 pages from YouTube.
Most marathon advice online is outdated or flat-out wrong. I'm a sports scientist, physiotherapist, and former professional triathlete. And in this video, I'll show you exactly how to train, fuel, pace, and plan your week to get the fastest time possible. Based on real science, not guesswork. So what does it actually take to run a f

In [24]:
from langchain.document_loaders import WebBaseLoader

urls = [
    "https://run.outsideonline.com/training/training-plans/marathon/a-16-week-marathon-training-plan-to-go-the-distance?scope=anon"
]

loader = WebBaseLoader(urls)
web_pages = loader.load()

print(f"Loaded {len(web_pages)} webpages.")
print(web_pages[0].page_content[:500])
print(web_pages[0].metadata)  # has source URL, title (if found), etc.

Loaded 1 webpages.





















16 Week Marathon Training Plan to Go the Distance




































 


































RUN | Powered by Outside
Powered by Outside



















Home




Featured



2025 UTMB Mont-Blanc


News


2025 Training Plan Central


Gear


Training


Newsletter


About Us



More 









































A 16-Week Marathon Training Plan to Go the Distance

If you're going to run a 26.2-mile race, you'll need a marathon trainin
{'source': 'https://run.outsideonline.com/training/training-plans/marathon/a-16-week-marathon-training-plan-to-go-the-distance?scope=anon', 'title': '16 Week Marathon Training Plan to Go the Distance', 'description': "If you're going to run a 26.2-mile race, you'll need a marathon training plan that incorporates proper strength training and recovery.", 'language': 'en-US'}
